In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
###################################################################
############################################################
# Imports
############################################################

import os
import random
import numpy as np
import pandas as pd
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as L
from pytorch_lightning.callbacks import ModelCheckpoint

import torchmetrics
from torchvision.models import resnet18


############################################################
# Seed Configuration (Reproducibility)
############################################################

SEED = 42

# python random
random.seed(SEED)

# numpy
np.random.seed(SEED)

# pytorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# deterministic GPU behaviour
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Lightning helper (covers dataloader workers too)
L.seed_everything(SEED, workers=True)


############################################################
# DataLoader worker seed function
############################################################

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

generator = torch.Generator()
generator.manual_seed(SEED)

Seed set to 42


# Loading songs path data in dict structure

In [3]:
################################################################
#

def load_genre_stems(root_path):

    data = {}

    genres = os.listdir(root_path)

    for genre in genres:

        genre_path = os.path.join(root_path, genre)

        if not os.path.isdir(genre_path):
            continue

        data[genre] = {}

        songs = os.listdir(genre_path)

        for song in songs:

            song_path = os.path.join(genre_path, song)

            if not os.path.isdir(song_path):
                continue

            stems = {}

            for file in os.listdir(song_path):

                if file.endswith(".wav"):

                    stem_name = file.replace(".wav","")

                    stems[stem_name] = os.path.join(song_path, file)

            data[genre][song] = stems

    return data

In [4]:
###########################
#
dataset_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

data = load_genre_stems(dataset_path)

In [5]:
#############################
#
print(data["rock"].keys())

dict_keys(['rock.00052', 'rock.00083', 'rock.00092', 'rock.00053', 'rock.00096', 'rock.00003', 'rock.00079', 'rock.00069', 'rock.00097', 'rock.00084', 'rock.00041', 'rock.00012', 'rock.00065', 'rock.00080', 'rock.00086', 'rock.00016', 'rock.00054', 'rock.00062', 'rock.00011', 'rock.00005', 'rock.00038', 'rock.00018', 'rock.00034', 'rock.00022', 'rock.00048', 'rock.00042', 'rock.00088', 'rock.00078', 'rock.00056', 'rock.00006', 'rock.00039', 'rock.00077', 'rock.00028', 'rock.00019', 'rock.00000', 'rock.00074', 'rock.00057', 'rock.00035', 'rock.00007', 'rock.00037', 'rock.00099', 'rock.00075', 'rock.00094', 'rock.00090', 'rock.00093', 'rock.00049', 'rock.00067', 'rock.00024', 'rock.00033', 'rock.00047', 'rock.00085', 'rock.00098', 'rock.00051', 'rock.00032', 'rock.00064', 'rock.00014', 'rock.00066', 'rock.00009', 'rock.00076', 'rock.00020', 'rock.00027', 'rock.00058', 'rock.00044', 'rock.00031', 'rock.00059', 'rock.00025', 'rock.00008', 'rock.00082', 'rock.00063', 'rock.00087', 'rock.000

In [6]:
####################################
#
print(data["rock"]["rock.00004"])

{'drums': '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00004/drums.wav', 'vocals': '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00004/vocals.wav', 'bass': '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00004/bass.wav', 'other': '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00004/other.wav'}


# Loading Noise data paths in dict structurre 

In [7]:
#######################################
#


def load_noise_dataset(csv_path, audio_dir):

    df = pd.read_csv(csv_path)

    noise_data = {}

    for _, row in df.iterrows():

        category = row["category"]
        filename = row["filename"]

        path = os.path.join(audio_dir, filename)

        if category not in noise_data:
            noise_data[category] = []

        noise_data[category].append(path)

    return noise_data

In [8]:
#####################################################
#
csv_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta/esc50.csv"
audio_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"

noise_data = load_noise_dataset(csv_path, audio_dir)

In [9]:
##################################################
#
print(len(noise_data))
print(noise_data["dog"][:3])

50
['/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/1-100032-A-0.wav', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/1-110389-A-0.wav', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/1-30226-A-0.wav']


In [10]:
##########################################
# checking samipling frequency 

# noise_sr = set()

# for category in noise_data:
    
#     for file in noise_data[category]:
        
#         _, sr = librosa.load(file, sr=None)
        
#         noise_sr.add(sr)

# print("Noise sampling rates:", noise_sr)

print("Noise sampling rates: {44100}")

Noise sampling rates: {44100}


In [11]:
################################################
#
# genre_sr = set()

# for genre in data:
    
#     for song in data[genre]:
        
#         for stem in data[genre][song]:
            
#             file = data[genre][song][stem]
            
#             _, sr = librosa.load(file, sr=None)
            
#             genre_sr.add(sr)

# print("Genre sampling rates:", genre_sr)
print("Genre sampling rates: {44100}")

Genre sampling rates: {44100}


# Data Spliting into train and validation 

In [12]:
######################################################
#


def split_dataset(genre_data, train_ratio=0.8):

    train_data = {}
    val_data = {}

    for genre in genre_data:

        songs = list(genre_data[genre].keys())

        random.shuffle(songs)

        split_idx = int(len(songs) * train_ratio)

        train_songs = songs[:split_idx]
        val_songs = songs[split_idx:]

        train_data[genre] = {song: genre_data[genre][song] for song in train_songs}
        val_data[genre] = {song: genre_data[genre][song] for song in val_songs}

    return train_data, val_data


In [13]:
###################################################
#
train_data, val_data = split_dataset(data)
for genre in train_data:
    print(
        genre,
        "train:", len(train_data[genre]),
        "val:", len(val_data[genre])
    )

disco train: 80 val: 20
metal train: 80 val: 20
reggae train: 80 val: 20
blues train: 80 val: 20
rock train: 80 val: 20
classical train: 80 val: 20
jazz train: 80 val: 20
hiphop train: 80 val: 20
country train: 80 val: 20
pop train: 80 val: 20


# Genarating  Traning data 

In [14]:
##################################################
#


SR = 22050
DURATION = 5
SAMPLES = SR * DURATION

In [15]:
#####################################################
#


def load_and_clip(path):

    audio, sr = librosa.load(path, sr=SR, mono=True)

    if len(audio) > SAMPLES:
        start = random.randint(0, len(audio) - SAMPLES)
        audio = audio[start:start + SAMPLES]

    else:
        pad = SAMPLES - len(audio)
        audio = np.pad(audio, (0, pad))

    return audio

In [16]:
#######################################################
#
def sample_stem_mashup(genre_data, genre):

    songs = random.sample(list(genre_data[genre].keys()), 4)

    stems = {
        "drums":  genre_data[genre][songs[0]]["drums"],
        "bass":   genre_data[genre][songs[1]]["bass"],
        "vocals": genre_data[genre][songs[2]]["vocals"],
        "other": genre_data[genre][songs[3]]["other"]
    }

    return stems

In [17]:
###########################################################
#
def sample_noise(noise_data):

    noise_class = random.choice(list(noise_data.keys()))
    noise_file = random.choice(noise_data[noise_class])

    noise = load_and_clip(noise_file)

    return noise

In [18]:
###########################################################
#
def create_mashup(genre_data, noise_data, genre):

    stems = sample_stem_mashup(genre_data, genre)

    drums = load_and_clip(stems["drums"])
    bass = load_and_clip(stems["bass"])
    vocals = load_and_clip(stems["vocals"])
    others = load_and_clip(stems["other"])  

    music_mix = (drums + bass + vocals + others) / 4.0

    noise = sample_noise(noise_data)

    final_audio = (music_mix + noise)/2.0
    max_val = np.max(np.abs(final_audio)) 
    if max_val > 0: 
        final_audio = final_audio / max_val

    return final_audio,stems

In [19]:
#############################################################
#

genre = random.choice(list(data.keys()))

audio,s = create_mashup(data, noise_data, genre)

print(audio.shape)

(110250,)


In [20]:
#############################################################
#

def audio_to_mel(audio):

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=2048,
        hop_length=512,
        n_mels=128
    )

    mel_db = librosa.power_to_db(mel)

    return mel_db

In [21]:
########################################################
#
genres = list(train_data.keys())

genre_to_idx = {g:i for i,g in enumerate(genres)}
idx_to_genre = {i:g for g,i in genre_to_idx.items()}

print(genre_to_idx)

{'disco': 0, 'metal': 1, 'reggae': 2, 'blues': 3, 'rock': 4, 'classical': 5, 'jazz': 6, 'hiphop': 7, 'country': 8, 'pop': 9}


# Data Loaders

In [22]:
###########################################################
#


class MashupDataset(Dataset):

    def __init__(self, genre_data, noise_data, size=450):

        self.samples = []
        used = set()

        genres = list(genre_data.keys())

        for genre in genres:

            count = 0

            while count < size:

                audio, stems = create_mashup(genre_data, noise_data, genre)

                key = (
                    stems["drums"],
                    stems["bass"],
                    stems["vocals"],
                    stems["other"]
                )

                if key in used:
                    continue

                used.add(key)

                mel = audio_to_mel(audio)

                mel = torch.from_numpy(mel).unsqueeze(0).float()

                label = genre_to_idx[genre]

                self.samples.append((mel, label))

                count += 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [23]:
##########################################################
#


train_dataset = MashupDataset(train_data, noise_data)
val_dataset = MashupDataset(val_data, noise_data,size=70)

In [35]:
###########################################################
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    worker_init_fn=seed_worker,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    worker_init_fn=seed_worker,
    num_workers=4
)

# Simple CNN Model

In [25]:
######################################################
#


class AudioCNN(L.LightningModule):

    def __init__(self, lr=1e-3, num_classes=10):

        super().__init__()

        self.lr = lr

        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.conv3 = nn.Conv2d(32,64,3,padding=1)

        self.pool = nn.MaxPool2d(2)

        # makes model independent of spectrogram width
        self.global_pool = nn.AdaptiveAvgPool2d((1,1))

        self.fc = nn.Linear(64,num_classes)

        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self,x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.global_pool(x)

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

    def training_step(self, batch, batch_idx):

        mel, label = batch

        pred = self(mel)

        loss = self.criterion(pred, label)

        acc = self.train_acc(pred, label)

        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):

        mel, label = batch

        pred = self(mel)

        loss = self.criterion(pred, label)

        acc = self.val_acc(pred, label)

        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):

        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)

        return optimizer

    def predict_step(self, batch, batch_idx):

        mel, sample_id = batch

        logits = self(mel)

        pred_class = torch.argmax(logits, dim=1)

        return pred_class, sample_id

# Resnet-18

In [26]:
####################################################################################################################



class AudioResNet18(L.LightningModule):

    def __init__(self, lr=1e-3, num_classes=10):

        super().__init__()

        self.lr = lr

        # load pretrained ResNet18
        self.model = resnet18(weights="IMAGENET1K_V1")

        # change first layer (RGB → 1 channel spectrogram)
        self.model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # change final classification layer
        self.model.fc = nn.Linear(512, num_classes)

        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = torchmetrics.Accuracy(
            task="multiclass", num_classes=num_classes
        )

        self.val_acc = torchmetrics.Accuracy(
            task="multiclass", num_classes=num_classes
        )

    def forward(self, x):

        return self.model(x)

    def training_step(self, batch, batch_idx):

        mel, label = batch

        pred = self(mel)

        loss = self.criterion(pred, label)

        acc = self.train_acc(pred, label)

        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):

        mel, label = batch

        pred = self(mel)

        loss = self.criterion(pred, label)

        acc = self.val_acc(pred, label)

        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):

        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr,weight_decay=1e-3)

        return optimizer

    def predict_step(self, batch, batch_idx):

        mel, sample_id = batch

        logits = self(mel)

        pred_class = torch.argmax(logits, dim=1)

        return pred_class, sample_id

# Efficient Net

In [27]:
##################################################################
from torchvision.models import efficientnet_b0


class AudioEfficientNet(L.LightningModule):

    def __init__(self, lr=5e-4, num_classes=10):
        super().__init__()

        self.lr = lr

        backbone = efficientnet_b0(weights="IMAGENET1K_V1")

        # adapt first conv layer for 1-channel spectrogram
        weight = backbone.features[0][0].weight.data
        backbone.features[0][0] = nn.Conv2d(
            1, 32, kernel_size=3, stride=2, padding=1, bias=False
        )
        backbone.features[0][0].weight.data = weight.mean(dim=1, keepdim=True)

        # replace classifier
        backbone.classifier[1] = nn.Linear(1280, num_classes)

        self.model = backbone

        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = torchmetrics.Accuracy(
            task="multiclass", num_classes=num_classes
        )

        self.val_acc = torchmetrics.Accuracy(
            task="multiclass", num_classes=num_classes
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):

        mel, label = batch

        logits = self(mel)

        loss = self.criterion(logits, label)

        acc = self.train_acc(logits, label)

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):

        mel, label = batch

        logits = self(mel)

        loss = self.criterion(logits, label)

        acc = self.val_acc(logits, label)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):

        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=1e-4
        )

        return optimizer

    def predict_step(self, batch, batch_idx):

        mel, sample_id = batch

        logits = self(mel)

        pred_class = torch.argmax(logits, dim=1)

        return pred_class, sample_id

In [28]:
####################################
MODEL_REGISTRY = {
    "cnn": AudioCNN,
    "resnet18": AudioResNet18,
    "efficientnet":AudioEfficientNet
}
selected = "efficientnet"

In [29]:
#########################################################
!pip install --upgrade wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 72.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: wandb
    Found existing installation: wandb 0.22.2
    Uninstalling wandb-0.22.2:
      Successfully uninstalled wandb-0.22.2


In [30]:
##################################################################
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = secret_value_0

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [36]:
##############################################
#


from pytorch_lightning.loggers import WandbLogger

# initialize wandb run
wandb.init(
    entity="indumiriyala-indian-institute-of-technology-madras",
    project="DL-21f3000478-notebook-t12026",
    config={
        "model": selected,
        "epochs": 50,
        "batch_size": 32
    }
)

# create lightning logger
wandb_logger = WandbLogger(
    experiment=wandb.run,
    log_model=True
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_acc",
    mode="max",
    save_top_k=1,
    filename="best-model",
    dirpath="checkpoints",
    verbose=True
)

model = MODEL_REGISTRY[selected]()

trainer = L.Trainer(
    max_epochs=50,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    log_every_n_steps=10,
    callbacks=[checkpoint_callback],
    logger=wandb_logger
)

trainer.fit(model, train_loader, val_loader)
wandb.finish()

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ EfficientNet       │  4.0 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss   │      0 │ train │     0 │
│ 2 │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_acc   │ MulticlassAccuracy │      0 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 4.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.0 M                                                                                                
Total estimated model params size (MB): 16                                                                         
Modules in train mode: 340                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Epoch 0, global step 313: 'val_acc' reached 0.51200 (best 0.51200), saving model to '/kaggle/working/checkpoints/best-model-v3.ckpt' as top 1
Epoch 1, global step 626: 'val_acc' reached 0.55000 (best 0.55000), saving model to '/kaggle/working/checkpoints/best-model-v3.ckpt' as top 1
Epoch 2, global step 939: 'val_acc' reached 0.55700 (best 0.55700), saving model to '/kaggle/working/checkpoints/best-model-v3.ckpt' as top 1
Epoch 3, global step 1252: 'val_acc' reached 0.60900 (best 0.60900), saving model to '/kaggle/working/checkpoints/best-model-v3.ckpt' as top 1
Epoch 4, global step 1565: 'val_acc' was not in top 1
Epoch 5, global step 1878: 'val_acc' was not in top 1
Epoch 6, global step 2191: 'val_acc' was not in top 1
Epoch 7, global step 2504: 'val_acc' was not in top 1
Epoch 8, global step 2817: 'val_acc' was not in top 1
Epoch 9, global step 3130: 'val_acc' was not in top 1
Epoch 10, global step 3443: 'val_acc' was not in top 1
Epoch 11, global step 3756: 'val_acc' was not in top

epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇████
train_acc,▅▁▅▄▅▅▅██▇██▇▇▇▅▇██▇▇██▅████████████▇███
train_loss,█▅▆▆▃▁▁▂▁▂▁▁▅▃▂▂▁▁▁▂▁▁▂▁▁▁▁▂▂▁▁▁▁▂▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
val_acc,▁▆▅▇▆▆▇▇▇▅▇▇█▆▇▆▇▆▆▇▅▆▇█▇▇█▆▇▆▅▇▇▇▇▇▅█▇▇
val_loss,▂▁▂▁▂▃▃▃▄▄▄▄▅▅▄▅▅▅▆▆▆▆▆▇▆▆▆▇▅▆▇▆█▆▅▆▇▇▆▆
epoch,59
train_acc,1
train_loss,0.05745
trainer/global_step,18779
val_acc,0.616


In [37]:
#########################################################
#
print(checkpoint_callback.best_model_path)

/kaggle/working/checkpoints/best-model-v3.ckpt


In [38]:
###############################################################
#
class TestDataset(Dataset):

    def __init__(self, test_df, audio_dir):

        self.test_df = test_df
        self.audio_dir = audio_dir

    def __len__(self):
        return len(self.test_df)

    def __getitem__(self, idx):

        row = self.test_df.iloc[idx]

        sample_id = row["id"]
        filename = row["filename"]

        path = f"{self.audio_dir}/{filename}"

        audio = load_and_clip(path)

        mel = audio_to_mel(audio)

        mel = torch.from_numpy(mel).unsqueeze(0).float()

        return mel, sample_id

In [39]:
###############################################################
#

test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

print(test_df.head())
test_dataset = TestDataset(test_df, "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    worker_init_fn=seed_worker,
    num_workers=4
)

   id              filename
0   1  mashups/song0001.wav
1   2  mashups/song0002.wav
2   3  mashups/song0003.wav
3   4  mashups/song0004.wav
4   5  mashups/song0005.wav


In [40]:
############################################################################
model = MODEL_REGISTRY[selected].load_from_checkpoint(checkpoint_callback.best_model_path)
model.eval()

AudioEfficientNet(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (sca

In [41]:
##############################################################################
trainer = L.Trainer(
    accelerator="gpu",
    devices=1
)

outputs = trainer.predict(model, test_loader)
predictions = []
ids = []

for batch in outputs:

    pred_class, sample_id = batch

    predictions.extend(pred_class.cpu().numpy())
    ids.extend(sample_id.cpu().numpy().tolist())
genres = [idx_to_genre[i] for i in predictions]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
2026-03-08 07:35:05.160650: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772955305.494942      99 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772955305.596235      99 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772955306.418026      99 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same 

Output()

In [42]:
########################## dummy ##############
import pandas as pd

submission = pd.DataFrame({
    "id": ids,
    "genre": genres
})

submission.to_csv("submission.csv", index=False)